# 01 - Explore the public data

Before any modelling: **do not skip this notebook.**  If the ethylene unit
value does not track the headline settlement series after a one-to-two month
lag, you have a unit error or the wrong product code, and every downstream
number will be wrong in a way that looks plausible.

Run `make data` first.

In [ ]:
import sys
from pathlib import Path

# work whether the kernel starts in the repo root or in notebooks/
root = next((p.parent for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ecpmodel').is_dir()), Path.cwd())
sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ecpmodel.features.build import MMBTU_TO_MWH, build_panel
from ecpmodel.io import load_cfg, load_interim

pd.options.display.width = 120
plt.style.use('seaborn-v0_8-whitegrid')

## 1. What is in the cache

Ingestion never applies publication lags - that happens once, in
`features/build.py`.  Everything here is therefore "the latest vintage" and is
exactly what a naive (wrong) backtest would use.

In [ ]:
SERIES = ['eurusd', 'brent_usd_bbl', 'gas_usd_mmbtu', 'eua_eur_t',
          'uv_ethylene', 'uvmed_ethylene', 'uv_propylene', 'uv_butadiene', 'uv_naphtha']

rows = []
for name in SERIES:
    s = load_interim(name)
    rows.append({'series': name, 'n': len(s),
                 'first': s.index.min() if len(s) else None,
                 'last': s.index.max() if len(s) else None,
                 'last_value': round(float(s.iloc[-1]), 3) if len(s) else None})

extra = sorted(p.stem for p in Path('data/interim').glob('ppi_*.parquet'))
for name in extra:
    s = load_interim(name)
    rows.append({'series': name, 'n': len(s), 'first': s.index.min(), 'last': s.index.max(),
                 'last_value': round(float(s.iloc[-1]), 3)})

pd.DataFrame(rows)

## 2. Brent in EUR/bbl

The ECB quotes **USD per EUR**, so divide the USD price by it.  Getting this
backwards is the single most common data bug in this kind of model.

In [ ]:
brent = load_interim('brent_usd_bbl')
fx = load_interim('eurusd')
gas = load_interim('gas_usd_mmbtu')

brent_eur = brent / fx.reindex(brent.index).ffill()
gas_eur_mwh = gas / fx.reindex(gas.index).ffill() / MMBTU_TO_MWH

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(brent.index, brent, label='Brent USD/bbl', lw=1)
axes[0].plot(brent_eur.index, brent_eur, label='Brent EUR/bbl', lw=1.4)
axes[0].set_ylabel('USD or EUR / bbl')
axes[0].legend(frameon=False, fontsize=8)

axes[1].plot(gas_eur_mwh.index, gas_eur_mwh, color='#b5651d', lw=1.4)
axes[1].set_ylabel('gas EUR / MWh')
fig.suptitle('Cost drivers in EUR', y=1.0)

## 3. Ethylene unit value vs the headline settlements

This is the sanity check.  Expect:

* the unit value within a factor of roughly 1.5 of the contract range
  (customs value, mix and delivery basis put them apart);
* the same direction of travel with a one-to-two month lag;
* a level that is **not** 3x off - that would be a units error (100 kg vs t is
  the usual culprit) or the wrong HS code.

In [ ]:
uv = load_interim('uv_ethylene')
tonnes = load_interim('tonnes_ethylene')

try:
    from ecpmodel.ingest.headlines import load as load_headlines
    h = load_headlines()
    heads = pd.Series(h['y'].to_numpy(), index=pd.DatetimeIndex(h['month']))
except Exception as exc:                                    # no headline file yet
    heads = pd.Series(dtype=float)
    print('no headlines:', exc)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(uv.index, uv, lw=1.4, label='Comext unit value (EUR/t)')
if len(heads):
    axes[0].scatter(heads.index, heads.to_numpy(), s=28, color='k', zorder=5,
                    label='ECP headline settlements')
axes[0].set_ylabel('EUR / t')
axes[0].legend(frameon=False, fontsize=8)

axes[1].semilogy(tonnes.index, tonnes, lw=1, color='#666')
axes[1].set_ylabel('trade volume, tonnes')
axes[1].set_title('Reported ethylene volume behind the unit value')

In [ ]:
# correlation of the unit value with the settlements at several lags
if len(heads) > 5:
    z = pd.concat([uv.rename('uv'), heads.rename('head')], axis=1).ffill()
    rows = []
    for k in range(4):
        rows.append({'comext_lag_months': k,
                     'corr_uv_vs_headline': z['uv'].corr(z['head'].shift(-k))})
    display(pd.DataFrame(rows).round(3))

## 4. Chemicals PPI

Weak evidence about ethylene specifically: it is a whole-economy index for
NACE C20 and is partly endogenous to ethylene itself.  It is also heavily
smoothed, which makes it look informative when it is not.

In [ ]:
ppi_files = sorted(Path('data/interim').glob('ppi_*.parquet'))
ppi = pd.read_parquet(ppi_files[0]).iloc[:, 0] if ppi_files else pd.Series(dtype=float)

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(ppi.index, ppi, color='#444', lw=1.4, label='chemicals PPI (NACE C20)')
if len(uv):
    ax.plot(uv.index, uv / uv.max() * ppi.max(), color='#12395b', lw=1,
            label='ethylene unit value, rescaled to the PPI axis')
ax.legend(frameon=False, fontsize=8)
ax.set_title('Chemicals PPI vs ethylene')

## 5. The as-of panel

This is the object the model consumes.  Availability lags from
`configs/data.yaml` are applied here - and nowhere else.

In [ ]:
cfg = load_cfg('data')
panel = build_panel()

X = panel['X']
print('panel months :', len(X), X.index[0].date(), '->', X.index[-1].date())
print('regressors   :', list(X.columns))
print()
panel['endog'].notna().sum().rename('n_observed').to_frame().T

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
for c in ('brent_eur', 'gas_eur_mwh', 'eua'):
    if c in panel['cost']:
        ax.plot(panel['cost'].index, panel['cost'][c], lw=1.2, label=c)
ax.set_title('Cost block used by the model (EUR units, published data only)')
ax.legend(frameon=False, fontsize=8)

## 6. Where the data stops

Each series has a different publication lag, so they end on different months.
This chart is the reason the backtest must be as-of rather than on the latest
vintage.

In [ ]:
lagd = cfg['availability_lag_days']
ends = []
for name, key in [('brent_usd_bbl', 'brent'), ('gas_usd_mmbtu', 'eu_gas'),
                  ('eua_eur_t', 'eua'), ('uv_ethylene', 'comext'),
                  (ppi_files[0].stem if ppi_files else 'ppi', 'ppi')]:
    s = load_interim(name)
    if len(s):
        ends.append({'series': name, 'lag_days': lagd.get(key), 'last_month': s.index.max()})
ends.append({'series': 'ECP headline', 'lag_days': lagd.get('headline'),
             'last_month': heads.index.max() if len(heads) else None})
display(pd.DataFrame(ends))

## 7. What to check before you model

1. Brent in EUR/bbl moves with EUR gas - if not, your FX sign is wrong.
2. The unit value tracks the settlements at lag 1-2. If the best correlation is
   at lag 0 or 3, check the product code before touching the model.
3. The unit value is within ~1.5x of the settlement range, not 3x off.
4. The PPI is smooth and slow. That is expected; do not read it as a signal.
5. The panel has at least 25-30 headline observations. Without them the model
   is a cost anchor plus smoothed trade data, not an estimate of the contract.